# Predicting Airline Satisfaction
### 02 · Preprocesamiento de datos

**Competencia:** [Kaggle · Playground Series S6E10](https://www.kaggle.com/competitions/playground-series-s6e10/overview)  
**Problema:** predecir la satisfacción de pasajeros mediante clasificación binaria.  
**Objetivo de este notebook:** preparar las características para entrenar y evaluar modelos de Machine Learning.

En esta etapa trabajamos con el conjunto etiquetado (`train.csv`), lo dividimos en entrenamiento y validación, tratamos los valores faltantes y codificamos las variables categóricas. **Las transformaciones se ajustan exclusivamente con los datos de entrenamiento**, para evitar que la información de validación influya en el aprendizaje.

Este notebook muestra el procedimiento **manualmente para comprenderlo**. En `03_modelado.ipynb` también utilizamos `Pipeline` y `ColumnTransformer` para automatizar el preprocesamiento.

## Guía de lectura

1. **Carga de datos:** lectura de los archivos y comprobación de sus dimensiones.
2. **Separación de datos:** creación de entrenamiento y validación.
3. **Valores faltantes:** imputación del retraso de llegada con la mediana.
4. **Variables categóricas:** conversión de texto a columnas numéricas con One-Hot Encoding.
5. **Conjunto final y exportación:** unión de características, controles de calidad y guardado para el siguiente notebook.

## 1. Carga de datos

Buscamos la carpeta principal del proyecto, tanto si Jupyter se ejecuta desde `notebooks/` como desde la raíz del repositorio. De esta forma evitamos rutas personales de Windows.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Localizar la carpeta raíz del proyecto.
carpeta_actual = Path.cwd()
if (carpeta_actual / "data" / "train.csv").is_file():
    proyecto = carpeta_actual
elif (carpeta_actual.parent / "data" / "train.csv").is_file():
    proyecto = carpeta_actual.parent
else:
    raise FileNotFoundError(
        "No se encontró data/train.csv. Ejecuta el notebook desde la raíz del proyecto "
        "o desde la carpeta notebooks/."
    )

ruta_train = proyecto / "data" / "train.csv"
ruta_test = proyecto / "data" / "test.csv"

train = pd.read_csv(ruta_train)
test = pd.read_csv(ruta_test)

print("Train:", train.shape)
print("Test:", test.shape)

Train: (699635, 23)
Test: (299844, 22)


`train.csv` contiene la variable objetivo `satisfaction`; `test.csv` no la incluye porque Kaggle espera que generemos esas predicciones al final del proyecto. **No utilizamos `test.csv` para ajustar transformaciones ni para entrenar modelos en este notebook.**

## 2. Separación entre entrenamiento y validación

Retiramos `id` porque identifica registros y no es una característica que queramos utilizar para predecir satisfacción. Separamos también la variable objetivo `satisfaction`.

Reservamos **20 % para validación** y utilizamos **80 % para entrenamiento**. Con `stratify=y` conservamos aproximadamente la proporción de pasajeros satisfechos e insatisfechos en ambos conjuntos. Usamos `random_state=42` para reproducir la partición.

In [2]:
# Variables predictoras y variable objetivo.
X = train.drop(columns=["satisfaction", "id"])
y = train["satisfaction"]

# Partición estratificada: 80 % entrenamiento y 20 % validación.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (559708, 21)
X_val: (139927, 21)
y_train: (559708,)
y_val: (139927,)


**Por qué separamos antes de imputar o codificar:** si calculáramos la mediana o las categorías usando todos los datos antes de dividirlos, incorporaríamos información del conjunto de validación. Queremos que ese conjunto sirva para evaluar un procedimiento que no lo haya utilizado durante su ajuste.

## 3. Tratamiento de valores faltantes

En el análisis exploratorio encontramos valores faltantes en **`Arrival Delay in Minutes`**. En vez de eliminar pasajeros, los reemplazamos por la **mediana del retraso de llegada calculada solamente con entrenamiento**. Luego aplicamos ese mismo valor a validación.

In [3]:
columna_retraso = "Arrival Delay in Minutes"

print("Faltantes antes de imputar:")
print("Entrenamiento:", X_train[columna_retraso].isna().sum())
print("Validación:", X_val[columna_retraso].isna().sum())

# Valor descriptivo: todavía no transforma ningún dato.
mediana_retraso = X_train[columna_retraso].median()
print("Mediana calculada en entrenamiento:", mediana_retraso)

Faltantes antes de imputar:
Entrenamiento: 227
Validación: 65
Mediana calculada en entrenamiento: 0.0


Utilizamos `SimpleImputer` para mantener un procedimiento claro y reutilizable. El método `fit()` aprende la mediana en entrenamiento; `transform()` reemplaza los valores faltantes **sin volver a calcularla**.

In [4]:
imputador = SimpleImputer(strategy="median")
imputador.fit(X_train[[columna_retraso]])

# Creamos copias para no modificar las particiones originales.
X_train = X_train.copy()
X_val = X_val.copy()

X_train[columna_retraso] = imputador.transform(
    X_train[[columna_retraso]]
).ravel()

X_val[columna_retraso] = imputador.transform(
    X_val[[columna_retraso]]
).ravel()

print("Mediana aprendida por SimpleImputer:", imputador.statistics_[0])
print("Faltantes totales en X_train:", X_train.isna().sum().sum())
print("Faltantes totales en X_val:", X_val.isna().sum().sum())

Mediana aprendida por SimpleImputer: 0.0
Faltantes totales en X_train: 0
Faltantes totales en X_val: 0


**Interpretación:** la mediana es una opción sencilla y poco sensible a valores extremos. En nuestro proyecto, los valores faltantes se concentran en una sola característica y el porcentaje es muy pequeño. La imputación permite conservar los pasajeros y mantener las mismas columnas en entrenamiento y validación.

## 4. Codificación de variables categóricas

Los modelos que utilizaremos necesitan variables numéricas. En nuestro conjunto identificamos cuatro variables de texto: `Gender`, `Customer Type`, `Type of Travel` y `Class`.

Con **One-Hot Encoding**, cada categoría se transforma en una columna con valores 0 o 1. Ajustamos el codificador en entrenamiento y después aplicamos la transformación a validación. Usamos `handle_unknown="ignore"` para evitar errores si más adelante aparece alguna categoría no vista durante el ajuste.

In [5]:
# Las cuatro columnas categóricas identificadas en el EDA.
columnas_categoricas = pd.Index([
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
])

print("Columnas categóricas:", columnas_categoricas.tolist())
print("Cantidad:", len(columnas_categoricas))

codificador = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Aprender categorías únicamente en entrenamiento.
codificador.fit(X_train[columnas_categoricas])

print("Categorías aprendidas:")
for nombre, categorias in zip(columnas_categoricas, codificador.categories_):
    print(f"{nombre}: {list(categorias)}")

Columnas categóricas: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Cantidad: 4
Categorías aprendidas:
Gender: ['Female', 'Male']
Customer Type: ['Loyal Customer', 'disloyal Customer']
Type of Travel: ['Business travel', 'Personal Travel']
Class: ['Business', 'Eco', 'Eco Plus']


El codificador produce **nueve columnas binarias** a partir de las cuatro variables de texto. Como las categorías se aprenden en entrenamiento, las columnas generadas tendrán el mismo nombre y orden cuando transformemos validación.

In [6]:
# Codificar las variables categóricas de ambos conjuntos.
categoricas_train = codificador.transform(X_train[columnas_categoricas])
categoricas_val = codificador.transform(X_val[columnas_categoricas])

nombres_nuevos = codificador.get_feature_names_out(columnas_categoricas)

# Conservamos los índices originales para unir las columnas correctamente.
categoricas_train_df = pd.DataFrame(
    categoricas_train,
    columns=nombres_nuevos,
    index=X_train.index
)

categoricas_val_df = pd.DataFrame(
    categoricas_val,
    columns=nombres_nuevos,
    index=X_val.index
)

print("Categorías codificadas en entrenamiento:", categoricas_train_df.shape)
print("Categorías codificadas en validación:", categoricas_val_df.shape)
print("\nNuevas columnas:", nombres_nuevos.tolist())

categoricas_train_df.head()

Categorías codificadas en entrenamiento: (559708, 9)
Categorías codificadas en validación: (139927, 9)

Nuevas columnas: ['Gender_Female', 'Gender_Male', 'Customer Type_Loyal Customer', 'Customer Type_disloyal Customer', 'Type of Travel_Business travel', 'Type of Travel_Personal Travel', 'Class_Business', 'Class_Eco', 'Class_Eco Plus']


,Gender_Female,Gender_Male,Customer Type_Loyal Customer,Customer Type_disloyal Customer,Type of Travel_Business travel,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus
159366,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
659958,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
434666,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
427957,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
409049,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


**Detalle importante:** mantenemos `index=X_train.index` y `index=X_val.index` al crear los DataFrames. Así, al unir las columnas numéricas y categóricas, las filas siguen correspondiendo a los mismos pasajeros. Si ignoráramos los índices podríamos mezclar registros accidentalmente.

## 5. Construcción de los conjuntos finales

Las características numéricas ya están listas después de imputar los retrasos. Las unimos con las nueve columnas creadas por One-Hot Encoding.

Partimos de **17 variables numéricas** y **4 categóricas**. Después de la codificación obtenemos **26 columnas** para los modelos: 17 numéricas y 9 binarias.

In [7]:
# Separar las columnas numéricas y unirlas con las categóricas codificadas.
numericas_train = X_train.drop(columns=columnas_categoricas)
numericas_val = X_val.drop(columns=columnas_categoricas)

X_train_final = pd.concat(
    [numericas_train, categoricas_train_df],
    axis=1
)

X_val_final = pd.concat(
    [numericas_val, categoricas_val_df],
    axis=1
)

print("Columnas numéricas:", numericas_train.shape[1])
print("Columnas de One-Hot Encoding:", categoricas_train_df.shape[1])
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)

X_train_final.head()

Columnas numéricas: 17
Columnas de One-Hot Encoding: 9
X_train_final: (559708, 26)
X_val_final: (139927, 26)


,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Arrival Delay in Minutes,Gender_Female,Gender_Male,Customer Type_Loyal Customer,Customer Type_disloyal Customer,Type of Travel_Business travel,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus
159366,50,2454,1,1,1,1,4,5,5,5,...,29.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
659958,54,1846,1,2,2,2,1,2,1,1,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
434666,31,1678,1,3,2,2,4,2,4,4,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
427957,22,759,4,4,5,4,5,5,5,5,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
409049,21,983,2,2,2,1,3,2,5,3,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


### Comprobaciones antes del modelado

Verificamos que ambos conjuntos tengan las mismas columnas y el mismo orden, que no queden valores faltantes y que cada fila conserve su correspondencia con la variable objetivo.

In [8]:
# Controles de consistencia para evitar errores en el notebook de modelado.
assert X_train_final.columns.equals(X_val_final.columns), "Las columnas de train y validación no coinciden."
assert X_train_final.index.equals(y_train.index), "Los índices de X_train e y_train no coinciden."
assert X_val_final.index.equals(y_val.index), "Los índices de X_val e y_val no coinciden."
assert not X_train_final.isna().any().any(), "Hay valores faltantes en entrenamiento."
assert not X_val_final.isna().any().any(), "Hay valores faltantes en validación."

print("Columnas compatibles:", X_train_final.columns.equals(X_val_final.columns))
print("Valores faltantes en entrenamiento:", X_train_final.isna().sum().sum())
print("Valores faltantes en validación:", X_val_final.isna().sum().sum())
print("Tipos de dato en entrenamiento:")
print(X_train_final.dtypes.value_counts())

Columnas compatibles: True
Valores faltantes en entrenamiento: 0
Valores faltantes en validación: 0
Tipos de dato en entrenamiento:
int64      16
float64    10
Name: count, dtype: int64


## 6. Guardar los datos procesados

Guardamos los conjuntos preparados en `data/processed/datos_preprocesados.pkl` para utilizarlos en `03_modelado.ipynb`. Conservamos también el imputador, el codificador y los nombres de las columnas categóricas.

Usamos `pickle` por comodidad dentro de este proyecto local. **No se recomienda abrir archivos `.pkl` de fuentes desconocidas**, porque pueden ejecutar código al cargarlos.

In [9]:
ruta = proyecto / "data" / "processed"
ruta.mkdir(parents=True, exist_ok=True)

datos = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "y_train": y_train,
    "y_val": y_val,
    "imputador": imputador,
    "codificador": codificador,
    "columnas_categoricas": columnas_categoricas
}

archivo = ruta / "datos_preprocesados.pkl"
pd.to_pickle(datos, archivo)

print("Archivo generado correctamente:", archivo.exists())
print("Ubicación dentro del proyecto:", archivo.relative_to(proyecto))

Archivo generado correctamente: True
Ubicación dentro del proyecto: data\processed\datos_preprocesados.pkl


## 7. Conclusiones

- Separamos los datos etiquetados en entrenamiento y validación antes de aplicar cualquier transformación.
- Tratamos los valores faltantes de `Arrival Delay in Minutes` con la mediana aprendida en entrenamiento.
- Convertimos las cuatro variables categóricas en nueve columnas mediante One-Hot Encoding.
- Obtuvimos conjuntos con **26 características**, sin valores faltantes y con las mismas columnas en entrenamiento y validación.
- Guardamos los resultados para continuar con el entrenamiento de regresión logística y Random Forest en `03_modelado.ipynb`.

**Nota sobre el proyecto:** este notebook muestra el preprocesamiento paso a paso para entender cada operación. Durante el modelado también construimos Pipelines, que reúnen estas transformaciones y permiten aplicarlas de forma consistente durante la validación cruzada y las predicciones finales.